# A Minimal Neural Network
### Classifying Iris Flowers

In this notebook we build and train a small neural network from scratch.

**Goal:** predict the species of an iris flower from four measurements.

**What you will see:**
1. Load data
2. Define a network
3. Train with gradient descent
4. Evaluate accuracy

No heavy abstractions — every step is visible.

## 1. Imports

In [ ]:
import torch
import torch.nn as nn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

## 2. The Data

The **Iris dataset** has 150 flowers, each described by 4 numbers:
- sepal length, sepal width, petal length, petal width

Each flower belongs to one of 3 species (classes 0, 1, 2).

In [ ]:
# Load the dataset
iris = load_iris()
X = iris.data    # shape: (150, 4)  -- 150 flowers, 4 features each
y = iris.target  # shape: (150,)    -- the correct class for each flower

print("Features shape:", X.shape)
print("Labels shape: ", y.shape)
print("Classes:      ", iris.target_names)

## 2b. Data Exploration

Before training anything, it is worth understanding what the data actually looks like.

- Are the classes well-separated?
- Which features carry the most information?
- Are the feature scales similar?

Answering these questions helps us choose a sensible architecture and spot potential problems early.

In [ ]:
import numpy as np

# Per-class mean and standard deviation for each feature
print("=== Per-species feature statistics ===\n")
header = f"{'Feature':<26}" + "".join(f"{name:>18}" for name in iris.target_names)
print(header)
print("-" * (26 + 18 * 3))

for i, feat in enumerate(iris.feature_names):
    row = f"{feat:<26}"
    for cls in range(3):
        mask = y == cls
        m, s = X[mask, i].mean(), X[mask, i].std()
        row += f"  {m:.2f} ± {s:.2f}    "
    print(row)

# Class distribution
print("\n=== Class distribution ===")
for cls, name in enumerate(iris.target_names):
    print(f"  {name}: {(y == cls).sum()} samples")

In [ ]:
colors  = ['#e74c3c', '#2ecc71', '#3498db']
markers = ['o', 's', '^']
pairs   = [(2, 3), (0, 2), (0, 3), (1, 2)]   # most informative pairs first

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

for ax, (i, j) in zip(axes.flat, pairs):
    for cls in range(3):
        mask = y == cls
        ax.scatter(X[mask, i], X[mask, j],
                   c=colors[cls], marker=markers[cls],
                   label=iris.target_names[cls],
                   alpha=0.75, edgecolors='k', linewidths=0.4, s=40)
    ax.set_xlabel(iris.feature_names[i], fontsize=9)
    ax.set_ylabel(iris.feature_names[j], fontsize=9)
    ax.legend(fontsize=7)
    ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle("Iris — Feature Scatter Plots", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Observation: petal length & petal width separate the three species most clearly.")

In [ ]:
# Split into training (80%) and test (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Convert to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test  = torch.tensor(y_test,  dtype=torch.long)

print("Training samples:", X_train.shape[0])
print("Test samples:    ", X_test.shape[0])

## 🎯🎯🎯 YOUR TASK 🎯🎯🎯 3. The Network

Our network has three layers:

```
Input (4)  -->  Hidden (16)  -->  Output (3)
```

- **Input layer:** one node per feature (4 features)
- **Hidden layer:** 16 nodes with ReLU activation  
  *(ReLU simply zeros out negative values: `max(0, x)`)*
- **Output layer:** one node per class (3 classes)

The output layer produces raw scores (logits). The highest score is the predicted class.

In [ ]:
class IrisNet(nn.Module):
    def __init__(self):
        super().__init__()
        # 🎯🎯🎯 TODO 🎯🎯🎯
        # Hint: get all these functions from the nn module
        self.layer1 = ...   # Linear Layer, 4 inputs  -> 16 hidden nodes
        self.relu   = ...
        self.layer2 = ...   # Linear layer, 16 hidden -> 3 output nodes

    def forward(self, x):
        x = self.layer1(x)   # linear transformation
        x = self.relu(x)     # non-linearity
        # 🎯🎯🎯 TODO 🎯🎯🎯
        x = ...              # produces 3 scores (one per class)
        return x

model = IrisNet()
print(model)

## 3b. Visualising the Network

The diagram below shows every node and every weight connection in **IrisNet**.

- **Blue nodes** — input features (4)
- **Orange nodes** — hidden layer with ReLU (16)
- **Green nodes** — output scores, one per class (3)

Each line represents a learnable weight. The network has
`4×16 + 16 + 16×3 + 3 = 115` trainable parameters in total.

In [ ]:
# @title Let's see how the network that you created looks like {display-mode: 'form'}
def draw_network(layer_sizes, layer_labels, layer_colors, ax):
    """Draw a fully-connected neural network diagram."""
    n_layers = len(layer_sizes)
    v_margin = 0.05   # vertical margin around nodes

    # Pre-compute node (x, y) positions
    positions = []
    for l, n in enumerate(layer_sizes):
        x = l / (n_layers - 1)
        ys = np.linspace(v_margin, 1 - v_margin, n)
        positions.append(list(zip([x] * n, ys)))

    # Draw edges first (so nodes sit on top)
    for l in range(n_layers - 1):
        for x1, y1 in positions[l]:
            for x2, y2 in positions[l + 1]:
                ax.plot([x1, x2], [y1, y2], color='#aaaaaa',
                        linewidth=0.4, alpha=0.35, zorder=1)

    # Draw nodes
    radius = 0.022
    for l, (nodes, color) in enumerate(zip(positions, layer_colors)):
        for (x, y) in nodes:
            circle = plt.Circle((x, y), radius, color=color,
                                 ec='#333333', linewidth=0.7, zorder=2)
            ax.add_patch(circle)

    # Layer labels and activation annotations
    activations = ['', 'ReLU', 'Softmax\n(at inference)']
    for l, (label, n, act) in enumerate(zip(layer_labels, layer_sizes, activations)):
        x = l / (n_layers - 1)
        ax.text(x, -0.04, f"{label}\n({n} nodes)",
                ha='center', va='top', fontsize=9, fontweight='bold')
        if act:
            ax.text(x, 1.05, act, ha='center', va='bottom',
                    fontsize=8, color='#555555', style='italic')

    ax.set_xlim(-0.12, 1.12)
    ax.set_ylim(-0.18, 1.15)
    ax.set_aspect('equal')
    ax.axis('off')

fig, ax = plt.subplots(figsize=(10, 6))
draw_network(
    layer_sizes  = [4, 16, 3],
    layer_labels = ['Input', 'Hidden', 'Output'],
    layer_colors = ['#3498db', '#e67e22', '#2ecc71'],
    ax           = ax
)
ax.set_title("IrisNet — Architecture Overview", fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

total_params = sum(p.numel() for p in model.parameters())
print(f"Total trainable parameters: {total_params}")

## 4. 🎯🎯🎯 YOUR TASK 🎯🎯🎯 Loss and Optimizer

**Loss function:** Cross-Entropy Loss  
Measures how wrong the network's predictions are. Lower is better.

**Optimizer:** SGD (Stochastic Gradient Descent)  
Updates the network's weights to reduce the loss.

The **learning rate** controls how large each update step is.

In [ ]:
# 🎯🎯🎯 TODO 🎯🎯🎯
loss_fn   = ...
optimizer = ..

## 5. 🎯🎯🎯 YOUR TASK 🎯🎯🎯 Training

One **epoch** = one full pass over all training data.

Each step:
1. **Forward pass** — compute predictions
2. **Compute loss** — how wrong are we?
3. **Backward pass** — compute gradients
4. **Update weights** — take one SGD step

In [ ]:
num_epochs = 200
loss_history = []  # we will plot this later

for epoch in range(num_epochs):
    # --- Step 1: Forward pass ---
    # 🎯🎯🎯 YOUR TASK 🎯🎯🎯
    predictions = ...

    # --- Step 2: Compute loss ---
    # 🎯🎯🎯 YOUR TASK 🎯🎯🎯
    loss = ...

    # --- Step 3: Backward pass ---
    # 🎯🎯🎯 YOUR TASK 🎯🎯🎯
    ...   # clear old gradients
    ...   # compute new gradients (backpropagation)

    # --- Step 4: Update weights ---
    # 🎯🎯🎯 YOUR TASK 🎯🎯🎯
    ...

    loss_history.append(loss.item())

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:3d} / {num_epochs}  |  Loss: {loss.item():.4f}")

## 6. Learning Curve

The loss should decrease over time — the network is learning.

In [ ]:
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss over Time")
plt.grid(True)
plt.show()

## 7. 🎯🎯🎯 YOUR TASK 🎯🎯🎯 Evaluation

Now we test on the held-out data the model has **never seen**.

In [ ]:
with torch.no_grad():   # no gradient computation needed for evaluation
    test_scores = ...
    predicted   = ...   # pick the class with highest score

correct  = (predicted == y_test).sum().item()
total    = y_test.shape[0]
accuracy = ...  # Percentage of correct guesses

print(f"Correct: {correct} / {total}")
print(f"Accuracy: {accuracy:.1f}%")

## 8. One Prediction Up Close

Let's look at what the network actually produces for a single flower.

In [ ]:
flower = X_test[0].unsqueeze(0)   # shape: (1, 4)
true_label = y_test[0].item()

with torch.no_grad():
    scores = model(flower)         # raw scores, shape: (1, 3)

# Convert scores to probabilities with softmax
probs = torch.softmax(scores, dim=1)[0]

print("True species:   ", iris.target_names[true_label])
print()
for i, name in enumerate(iris.target_names):
    print(f"  P({name:15s}) = {probs[i].item():.3f}")

---
## 9. 🎯🎯🎯 YOUR TASK (bonus!) 🎯🎯🎯 — Compare SGD vs Adam

SGD updates weights using only the gradient scaled by a fixed learning rate.
**Adam** keeps a running estimate of the first and second moment of the gradients,
giving each parameter its own adaptive learning rate. It often converges faster and
with less sensitivity to the initial learning rate.

**Your task:** train a second, identical network using the Adam optimizer and plot
both loss curves on the same axes to compare convergence speed.

In [ ]:
# --- Train a fresh model with Adam ---
model_adam = IrisNet()
optimizer_adam = torch.optim.Adam(model_adam.parameters(), lr=0.01)

loss_history_adam = []
for epoch in range(num_epochs):
    # 🎯🎯🎯 TODO 🎯🎯🎯
    ...

# --- Plot both curves ---
plt.figure(figsize=(8, 4))
plt.plot(loss_history,      label='SGD  (lr=0.05)', linewidth=1.5)
plt.plot(loss_history_adam, label='Adam (lr=0.01)', linewidth=1.5, linestyle='--')
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("SGD vs Adam — Loss Comparison")
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

# Final accuracy of Adam model
with torch.no_grad():
    acc_adam = (model_adam(X_test).argmax(1) == y_test).float().mean().item() * 100
print(f"Adam model test accuracy: {acc_adam:.1f}%")

---
## 10 — Confusion Matrix

Accuracy is a single number, but sometimes it hides *where* the model makes mistakes.
A **confusion matrix** shows, for every true class, how often each class was predicted.

- The diagonal holds correct predictions.
- Off-diagonal cells reveal which pairs of species the model confuses.

Run this cell to plot the confusion matrix for the test set.

In [ ]:
from sklearn.metrics import confusion_matrix

# Get predictions on the test set (reuse the original SGD model)
with torch.no_grad():
    predicted_labels = model(X_test).argmax(1).numpy()
true_labels = y_test.numpy()

# Build the 3×3 confusion matrix
cm = confusion_matrix(true_labels, predicted_labels)

# Plot
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')

ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(iris.target_names, rotation=30, ha='right')
ax.set_yticklabels(iris.target_names)
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title("Confusion Matrix — Test Set")

# Annotate cells
for i in range(3):
    for j in range(3):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                fontsize=13, fontweight='bold', color=color)

plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

---
## Recap

| Step | What we did |
|------|-------------|
| Data | Loaded Iris, split into train/test |
| Model | 2-layer network: Linear → ReLU → Linear |
| Loss | Cross-entropy (for multi-class classification) |
| Training | 200 epochs of SGD |
| Evaluation | Accuracy on held-out test set |

**Things to try:**
- Set different learning rates `lr` (e.g. `0.1` or `0.01`) — what happens to training?
- Change the hidden layer size (`16` → `4` or `64`)
- Add a second hidden layer
- Train for more or fewer epochs